# Notebook 02 — Feature Engineering & Labels

**Projet :** Juste des Ventilateurs — M2 Data/IA LaPlateforme_  
**Phase :** 3 — Feature engineering et préparation des données supervisées  
**Objectif :** Explorer les features calculées et les labels de supervision.

## Pipeline features
```
data/raw/episode=XXX/          (données brutes — Notebook 01)
    └─► features/pipeline.py
        ├─► temporal.py        → deltas, rolling stats, margin
        ├─► contextual.py      → hot_zone, shutdowns, recovering
        ├─► energy.py          → PUE, power_fans, fan_energy_ratio
        └─► labeler.py         → failure_30s, failure_60s, action_class
            └─► data/processed/episode=XXX/features.parquet
```

## Groupes de features (47 features utilisées pour la modélisation)
| Groupe | Exemples | Rôle |
|--------|---------|------|
| Temporelles | `temp_delta_30s`, `margin_to_shutdown`, `rpm_rolling_mean_30s` | Dynamique thermique |
| Contextuelles | `time_in_hot_zone_s`, `nb_shutdowns_episode`, `is_recovering` | Historique machine |
| Energétiques | `pue_estimated`, `power_fans_w`, `fan_energy_ratio` | Consommation |

## Labels de supervision
| Label | Description | Utilisation |
|-------|-------------|-------------|
| `failure_30s` | Panne dans les 30s | Prédiction court-terme |
| `failure_60s` | Panne dans les 60s | Prédiction moyen-terme (Phase 4) |
| `hot_30s` | Température critique dans 30s | Alerte thermique |
| `action_class` | RPM optimal {0,1,2,3,4} → {0,1500,2500,3500,4500} | Contrôleur supervisé (Phase 5) |

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

_root = Path.cwd()
for _candidate in [_root, _root.parent, _root.parent.parent]:
    if (_candidate / 'features').exists() and (_candidate / 'data').exists():
        _root = _candidate
        break
os.chdir(_root)
sys.path.insert(0, str(_root))

PROC_DIR = Path('data/processed')
episodes = sorted(p for p in PROC_DIR.glob('episode=*') if p.is_dir())
print(f'Repertoire : {Path.cwd()}')
print(f'Episodes processed : {[p.name for p in episodes]}')

## 1. Schéma des features calculées

In [ ]:
df = pd.read_parquet(episodes[0] / 'features.parquet')
print(f'Shape (episode=001) : {df.shape}')
print(f'Machines : {df["machine_id"].unique().tolist()}')
print()

# Groupes de features
FEATURE_GROUPS = {
    'Brutes (raw)': ['temperature_c','sensor_temp_max','sensor_temp_mean',
                     'power_w','energy_kwh','fan_rpm_mean','load_estimated'],
    'Temporelles': ['temp_delta_5s','temp_delta_30s','temp_rolling_mean_30s',
                    'temp_rolling_mean_60s','temp_rolling_std_30s',
                    'margin_to_shutdown','margin_pct','margin_delta_30s',
                    'rpm_rolling_mean_30s','load_rolling_mean_30s'],
    'Contextuelles': ['time_in_hot_zone_s','nb_shutdowns_episode',
                      'nb_degraded_episode','ticks_since_last_shutdown',
                      'has_fan_fault','has_power_surge','is_recovering'],
    'Energetiques': ['power_fans_w','fan_energy_ratio','pue_estimated',
                     'energy_fans_kwh_cumulated','pue_rolling_mean_30s'],
    'Labels': ['failure_30s','failure_60s','hot_30s','action_class','optimal_rpm'],
}

for group, cols in FEATURE_GROUPS.items():
    present = [c for c in cols if c in df.columns]
    print(f'[{group}] ({len(present)} features)')
    for c in present:
        null_pct = df[c].isnull().mean() * 100
        print(f'  {c:<40} mean={df[c].mean():.2f}  null={null_pct:.0f}%')
    print()

## 2. Distribution des labels par épisode

In [ ]:
label_cols = ['failure_30s', 'failure_60s', 'hot_30s']
ep_labels = []
for ep in episodes:
    df_ep = pd.read_parquet(ep / 'features.parquet')
    meta_f = Path('data/raw') / ep.name / 'metadata.json'
    scenario = 'unknown'
    if meta_f.exists():
        import json
        scenario = json.loads(meta_f.read_text()).get('scenario','?')
    row = {'episode': ep.name.replace('episode=',''), 'scenario': scenario, 'n': len(df_ep)}
    for lbl in label_cols:
        if lbl in df_ep.columns:
            row[lbl] = df_ep[lbl].mean() * 100
        else:
            row[lbl] = 0.0
    ep_labels.append(row)

df_lbl = pd.DataFrame(ep_labels).set_index('episode')

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(df_lbl))
w = 0.25
colors_lbl = ['#e74c3c', '#c0392b', '#f39c12']
for i, (lbl, color) in enumerate(zip(label_cols, colors_lbl)):
    ax.bar(x + i*w, df_lbl[lbl], width=w, label=lbl, color=color, alpha=0.85, edgecolor='white')
ax.set_xticks(x + w)
ax.set_xticklabels([f"{ep}\n{sc}" for ep, sc in zip(df_lbl.index, df_lbl['scenario'])], fontsize=9)
ax.set_ylabel('% positifs')
ax.set_title('Distribution des labels par épisode')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('evaluation/results/fig_02_label_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
display(df_lbl[['scenario','n'] + label_cols].round(1))

## 3. Distribution de action_class

In [ ]:
action_to_rpm = {0: 0, 1: 1500, 2: 2500, 3: 3500, 4: 4500}
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes_flat = axes.flatten()

for ax, ep, row in zip(axes_flat, episodes, ep_labels):
    df_ep = pd.read_parquet(ep / 'features.parquet')
    if 'action_class' not in df_ep.columns:
        ax.set_visible(False)
        continue
    counts = df_ep['action_class'].value_counts().sort_index()
    labels = [f'{action_to_rpm.get(int(k), k)} RPM' for k in counts.index]
    colors_rpm = ['#95a5a6','#7f8c8d','#2980b9','#e67e22','#c0392b']
    ax.bar(labels, counts.values / len(df_ep) * 100,
           color=[colors_rpm[int(k)] for k in counts.index], alpha=0.85, edgecolor='white')
    ax.set_title(f"ep={row['episode']} ({row['scenario']})", fontsize=9)
    ax.set_ylabel('% du temps')
    ax.set_xlabel('RPM')
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle('Distribution de action_class (RPM optimal) par épisode', fontsize=12)
plt.tight_layout()
plt.savefig('evaluation/results/fig_02_action_class.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Corrélations des features avec failure_60s

In [ ]:
# Concat tous les épisodes
dfs_all = [pd.read_parquet(ep / 'features.parquet') for ep in episodes]
df_all = pd.concat(dfs_all, ignore_index=True)
print(f'Dataset complet : {df_all.shape}')

# Features numériques (hors labels et ids)
exclude = {'timestamp','cluster_id','machine_id','role','msg_type','status',
           'fan_modes','fault_types','failure_30s','failure_60s','hot_30s',
           'action_class','optimal_rpm','time_to_failure_s','time_in_degraded_s'}
num_cols = [c for c in df_all.select_dtypes(include='number').columns if c not in exclude]

corr = df_all[num_cols + ['failure_60s']].corr()['failure_60s'].drop('failure_60s').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top20 = corr.head(20)
bars = ax.barh(top20.index[::-1], top20.values[::-1], color='#3498db', alpha=0.8, edgecolor='white')
ax.set_xlabel('|Corrélation| avec failure_60s')
ax.set_title('Top 20 features corrélées avec failure_60s (tous épisodes)')
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='seuil 0.5')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)
for bar, val in zip(bars, top20.values[::-1]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('evaluation/results/fig_02_feature_correlations.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Split temporel — stratégie Option A

In [ ]:
from models.failure_prediction.splitter import TemporalSplitter

splitter = TemporalSplitter()
X_train, X_val, X_test, y_train, y_val, y_test = splitter.split(label_col='failure_60s')

print('Split Option A (70 / 15 / 15 par épisode)')
print(f'  X_train : {X_train.shape}  positifs={y_train.mean():.1%}')
print(f'  X_val   : {X_val.shape}  positifs={y_val.mean():.1%}')
print(f'  X_test  : {X_test.shape}  positifs={y_test.mean():.1%}')
print(f'  Features: {X_train.shape[1]}')
print()
print('Features utilisées (47) :')
print(list(X_train.columns))

# Vérification : pas de fuite temporelle
print()
print('Vérification anti-fuite : les features ne contiennent pas les labels')
label_leaks = [c for c in X_train.columns if 'failure' in c or c == 'action_class']
print(f'  Colonnes suspectes dans X_train : {label_leaks or "aucune"}')

## 6. Distributions des features clés

In [ ]:
key_features = [
    'temperature_c', 'margin_to_shutdown', 'fan_rpm_mean',
    'temp_rolling_mean_30s', 'pue_estimated', 'load_estimated'
]
key_features = [f for f in key_features if f in X_train.columns]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes_flat = axes.flatten()

for ax, feat in zip(axes_flat, key_features):
    vals_safe = X_train.loc[y_train == 0, feat].dropna()
    vals_danger = X_train.loc[y_train == 1, feat].dropna()
    bins = np.linspace(
        min(vals_safe.quantile(0.01), vals_danger.quantile(0.01)),
        max(vals_safe.quantile(0.99), vals_danger.quantile(0.99)),
        40
    )
    ax.hist(vals_safe,   bins=bins, alpha=0.6, color='#3498db', label='Sain (0)',    density=True)
    ax.hist(vals_danger, bins=bins, alpha=0.6, color='#e74c3c', label='Danger (1)',  density=True)
    ax.set_title(feat, fontsize=10)
    ax.set_ylabel('Densité')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Distributions des features clés — Sain vs Danger (failure_60s)', fontsize=12)
plt.tight_layout()
plt.savefig('evaluation/results/fig_02_feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Observations et conclusions

### Structure des features (66 colonnes processed, 47 pour la modélisation)
- **Temporelles (19)** : deltas et rolling stats sur température, RPM, puissance, margin
- **Contextuelles (13)** : historique de la machine (hot_zone, shutdowns, recovering)
- **Energétiques (6)** : PUE, consommation fans, ratio énergie
- **Exclues de la modélisation** : colonnes avec 100% NaN (`machines_total`, `machines_on`), colonnes d'identification, labels

### Labels
- `failure_60s` : ~20% positifs sur 5 épisodes / ~27% sur l'épisode stress — déséquilibre 1:4 à 1:5
- `hot_30s` : uniquement présent dans l'épisode `stress` (25.7%) — rare dans les conditions normales
- `action_class` : classe 0 (RPM=0) souvent majoritaire — politique oracle conservatrice en régime nominal

### Features les plus discriminantes
- `pue_estimated` et `pue_rolling_mean_30s` : corrélation ~1.0 avec failure_60s sur certains épisodes → attention au data leakage (le PUE est calculé à partir de l'état de charge qui prédit la panne)
- `margin_to_shutdown` : feature la plus interprétable — distance directe au seuil de shutdown
- Les rolling means thermiques (30s, 60s) sont de bons prédicteurs de tendance

### Split temporel
- Stratégie Option A : couper chaque épisode chronologiquement (70/15/15%), puis concaténer
- Garantit que tous les scénarios sont représentés dans train/val/test
- Évite la fuite temporelle intra-épisode

→ **Voir notebook 03** pour la modélisation de prédiction de pannes (Phase 4).